<a href="https://colab.research.google.com/github/Bindiya-Anand/IndiaKart_E-Commerce_Analytics/blob/main/Notebooks/Phase1_DataCleaning_Bindiya_Anand.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Internmo Data Analytics Project 1
## IndiaKart - E-Com
### Data Analyst Intern: Bindiya Anand
#### Intern Id: Inter563

# Phase 1: Data Cleaning & Quality Check

In [ ]:
# Importing necessary libraries
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [ ]:
#pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)

### Dataset 1: orders DA.csv

In [ ]:
# Loading CSV file: orders DA.csv
orders = pd.read_csv('/content/orders DA.csv')
orders.head()

,order_id,customer_id,order_date,order_time,status,city,state,pincode,total_amount,gst_amount,shipping_charge,discount_amount,final_amount,payment_method,shipping_partner,tracking_id,delivered_date,is_cod,channel
0,ORD000001,CUST02946,22-05-2025,17:10:00,Processing,Pune,Maharashtra,300862,100984.19,16364.64,0,20486.45,100984.19,Cash on Delivery,BlueDart,TRK354572763,NaN,1,App
1,ORD000002,CUST04232,16-10-2023,05:32:00,Delivered,Ludhiana,Punjab,387852,254478.01,41599.80,0,19338.79,254478.01,UPI,DTDC,TRK272838570,19-10-2023,0,Website
2,ORD000003,CUST08078,08-10-2023,12:32:00,Cancelled,Meerut,Uttar Pradesh,294014,32557.06,3562.92,0,696.86,32557.06,Debit Card,Amazon Logistics,TRK338505401,NaN,0,App
3,ORD000004,CUST09612,02-10-2023,08:05:00,Delivered,Nashik,Maharashtra,626047,24836.94,3092.88,0,4029.94,24836.94,UPI,DTDC,TRK289792017,09-10-2023,0,Mobile Web
4,ORD000005,CUST03065,15-06-2024,20:19:00,Delivered,Moradabad,Uttar Pradesh,409210,156689.29,25092.36,0,7805.07,156689.29,UPI,Ecom Express,TRK978240041,22-06-2024,0,App


In [ ]:
# orders dataset dimensions
orders.shape

(50000, 19)

- Number of records = 50000
- Number of Attributes = 19

In [ ]:
# orders dataset summary
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          50000 non-null  object 
 1   customer_id       50000 non-null  object 
 2   order_date        50000 non-null  object 
 3   order_time        50000 non-null  object 
 4   status            50000 non-null  object 
 5   city              50000 non-null  object 
 6   state             50000 non-null  object 
 7   pincode           50000 non-null  int64  
 8   total_amount      50000 non-null  float64
 9   gst_amount        50000 non-null  float64
 10  shipping_charge   50000 non-null  int64  
 11  discount_amount   50000 non-null  float64
 12  final_amount      50000 non-null  float64
 13  payment_method    50000 non-null  object 
 14  shipping_partner  50000 non-null  object 
 15  tracking_id       50000 non-null  object 
 16  delivered_date    32499 non-null  object

Observation:
- delivered_date has 17,501 null values. That means 35.002% of null values exist in this column.
- data type of delivered_date is object, which must be changed to datatime dtype.
- data type of order_date is object, which must be changed to datatime dtype.
- data type of order_time is object, which must be parsed into a time representation.

In [ ]:
# finding duplicate
# order_id uniqueness check
orders['order_id'].is_unique

True

In [ ]:
orders.duplicated().sum()

np.int64(0)

Observation:
order_id does not contain any duplicate values.

is_unique property:
- Returns True: If every single value in the object appears exactly once.
- Returns False: If there is even one duplicate value present.

In [ ]:
# tracking_id uniqueness check
orders['tracking_id'].is_unique

False

why there is dulplicate in tracking_id?

In [ ]:
orders['tracking_id'].duplicated().sum()

np.int64(1)

In [ ]:
orders.groupby('tracking_id')['order_id'].nunique().sort_values(ascending = False)

,order_id
tracking_id,
TRK726370663,2
TRK702163905,1
TRK702167811,1
TRK702171891,1
TRK702196807,1
...,...
TRK405127150,1
TRK405155082,1
TRK405161848,1


In [ ]:
orders[orders['tracking_id'].duplicated(keep=False)]

,order_id,customer_id,order_date,order_time,status,city,state,pincode,total_amount,gst_amount,shipping_charge,discount_amount,final_amount,payment_method,shipping_partner,tracking_id,delivered_date,is_cod,channel
10138,ORD010139,CUST02397,19-11-2023,09:25:00,Delivered,Lucknow,Uttar Pradesh,481001,15689.35,3747.52,0,1442.17,15689.35,Net Banking,Delhivery,TRK726370663,24-11-2023,0,Mobile Web
49124,ORD049125,CUST09373,24-04-2024,02:01:00,Delivered,Mysuru,Karnataka,175963,36041.62,4521.36,0,6157.74,36041.62,Debit Card,Amazon Logistics,TRK726370663,27-04-2024,0,App


Finding:
- One tracking_id is associated with two different orders.

Investigation:
- The two orders have different order IDs and substantially different order and delivery dates (approximately one year apart).
- Other order-related attributes are also different. Therefore, the repeated tracking ID is unlikely to represent a legitimate shared shipment.

In [ ]:
# converting attributes from object to datatime datatype

# delivered_date
orders['delivered_date'] = pd.to_datetime(orders['delivered_date'], format = '%d-%m-%Y')
print('Data type of delivered_date:', orders.delivered_date.dtype)

# order_date
orders['order_date'] = pd.to_datetime(orders['order_date'], format = '%d-%m-%Y')
print('Data type of order_date:', orders.order_date.dtype)

# order_time
orders['order_time'] = pd.to_datetime(orders['order_time'], format = '%H:%M:%S').dt.time
print('Data type of order_time:', orders.order_time.dtype)

Data type of delivered_date: datetime64[ns]
Data type of order_date: datetime64[ns]
Data type of order_time: object


In [ ]:
# checking whether the order_time has been sucessfully parsed into time object
type(orders['order_time'].iloc[0])

datetime.time

In [ ]:
# handling null values NaN delivered_date
# why the null values?
# expect for orders with status = delivered, remaining orders' delivered_date would be unknown until its delivered.

orders.groupby('status')['delivered_date'].apply(lambda x: x.isnull().sum())

,delivered_date
status,
Cancelled,5996
Delivered,0
Processing,2501
Returned,3933
Shipped,5071


Observation:
- The *Delivered* group doesn't have any null/missing values.
- The remaining status groups such as *Cancelled, Processing, Returned, Shipped* have null values because for some the orders are yet to be delivered while for others, the delivered_date is not necessary.

In [ ]:
# checking if delivery_date is after order_date

valid_count = 0
invalid_count = 0
missing_count = 0

for i in range(len(orders)):
  if pd.isnull(orders['delivered_date'].iloc[i]):
    missing_count += 1
  elif orders['delivered_date'].iloc[i] > orders['order_date'].iloc[i]:
    valid_count += 1
  else:
    invalid_count += 1

print("Flag counts")
print('Valid:', valid_count)
print('Missing:', missing_count)
print('Invalid:', invalid_count)

Flag counts
Valid: 32499
Missing: 17501
Invalid: 0


Observation:
- The records are flagged as Valid, Missing and Invalid depending on whether the delivered_date is after order_date.
- Missing is flagged for records where the delivered_date is NaN. Which means the order status is not marked as *Delivered* and corresponds to a different status group.
- There are 0 invalid records meaning all the orders are having delivered_date after the order_date.

In [ ]:
# checking for high value orders; orders above 5 lakhs
high_value_orders = orders[orders['final_amount'] > 500000]

len(high_value_orders)

312

Findings:
- There are 312 orders having *final_amount* above 5 lakhs.

In [ ]:
high_value_orders[
    ['order_id', 'customer_id', 'order_date',
     'status', 'final_amount']
]

,order_id,customer_id,order_date,status,final_amount
33,ORD000034,CUST01867,2024-03-10,Delivered,527178.05
34,ORD000035,CUST01370,2025-05-26,Cancelled,532061.13
306,ORD000307,CUST03874,2024-03-20,Cancelled,550716.17
583,ORD000584,CUST07624,2024-10-30,Delivered,745667.68
590,ORD000591,CUST05298,2025-05-22,Cancelled,520293.72
...,...,...,...,...,...
39194,ORD039195,CUST02704,2024-09-02,Delivered,588456.74
39582,ORD039583,CUST06723,2023-06-28,Delivered,519500.64
39793,ORD039794,CUST04753,2024-02-02,Delivered,640296.82
39804,ORD039805,CUST04071,2023-07-11,Delivered,796481.05


In [ ]:
orders.describe()

,order_date,pincode,total_amount,gst_amount,shipping_charge,discount_amount,final_amount,delivered_date,is_cod
count,50000,50000.000000,5.000000e+04,50000.000000,50000.000000,50000.000000,5.000000e+04,32499,50000.000000
mean,2024-06-24 04:51:35.999999744,450264.923840,6.294671e+04,9339.163771,1.002260,6798.490053,6.294771e+04,2024-06-28 16:05:16.366657536,0.117660
min,2023-06-24 00:00:00,100007.000000,8.944000e+01,0.000000,0.000000,0.050000,1.360700e+02,2023-06-26 00:00:00,0.000000
25%,2023-12-24 00:00:00,274830.250000,6.893520e+03,787.500000,0.000000,781.032500,6.893520e+03,2023-12-30 00:00:00,0.000000
50%,2024-06-24 00:00:00,450831.000000,2.418338e+04,3068.280000,0.000000,2560.345000,2.418338e+04,2024-06-28 00:00:00,0.000000
75%,2024-12-24 00:00:00,626660.000000,7.149574e+04,9381.880000,0.000000,7370.285000,7.149574e+04,2024-12-28 00:00:00,0.000000
max,2025-06-23 00:00:00,799995.000000,1.133382e+06,188058.240000,79.000000,132116.060000,1.133382e+06,2025-06-30 00:00:00,1.000000
std,NaN,202553.430112,9.789814e+04,15946.558659,7.675688,11244.628851,9.789750e+04,NaN,0.322208


### Dataset 2: order_items DA.csv

In [ ]:
# Loading CSV file: order_items DA.csv
import pandas as pd
items = pd.read_csv('/content/order_items DA.csv')
items.head()

,item_id,order_id,product_id,product_name,category,quantity,unit_price,gst_rate,gst_amount,discount_amount,total_price
0,ITM0000001,ORD000001,PRD0522,HCL Cricket Bat Ultra,Sports & Fitness,1,40989,12,4918.68,7861.45,38046.23
1,ITM0000002,ORD000001,PRD0094,Berger Router Pro,Electronics,1,62532,18,11255.76,12373.11,61414.65
2,ITM0000003,ORD000001,PRD0613,Voltas Educational Toy Standard,Toys & Baby,1,1585,12,190.20,251.89,1523.31
3,ITM0000004,ORD000002,PRD0556,Ambrane Cycle Standard,Sports & Fitness,3,1107,12,398.52,631.44,3088.08
4,ITM0000005,ORD000002,PRD0026,Pebble Tablet Premium,Electronics,2,114448,18,41201.28,18707.35,251389.93


In [ ]:
# order_items dataset dimensions
items.shape

(100000, 11)

- Number of Records: 100000
- Number of Attributes: 11

In [ ]:
# order_items dataset summary
items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 11 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   item_id          100000 non-null  object 
 1   order_id         100000 non-null  object 
 2   product_id       100000 non-null  object 
 3   product_name     100000 non-null  object 
 4   category         100000 non-null  object 
 5   quantity         100000 non-null  int64  
 6   unit_price       100000 non-null  int64  
 7   gst_rate         100000 non-null  int64  
 8   gst_amount       100000 non-null  float64
 9   discount_amount  100000 non-null  float64
 10  total_price      100000 non-null  float64
dtypes: float64(3), int64(3), object(5)
memory usage: 8.4+ MB


Observation:
- No missing / null values present is order_items DA.
- Data types of the attributes are fine. No change required.

In [ ]:
# finding duplicate
# item_id uniqueness check

items['item_id'].is_unique

True

- item_id contains all unique values.

In [ ]:
items['order_id'].is_unique

False

In [ ]:
items.groupby('order_id')['item_id'].nunique().sort_values(ascending = False)

,item_id
order_id,
ORD000009,4
ORD039929,4
ORD000011,4
ORD000014,4
ORD039901,4
...,...
ORD000012,1
ORD039923,1
ORD039906,1


In [ ]:
items[items['order_id'].duplicated(keep=False)]

,item_id,order_id,product_id,product_name,category,quantity,unit_price,gst_rate,gst_amount,discount_amount,total_price
0,ITM0000001,ORD000001,PRD0522,HCL Cricket Bat Ultra,Sports & Fitness,1,40989,12,4918.68,7861.45,38046.23
1,ITM0000002,ORD000001,PRD0094,Berger Router Pro,Electronics,1,62532,18,11255.76,12373.11,61414.65
2,ITM0000003,ORD000001,PRD0613,Voltas Educational Toy Standard,Toys & Baby,1,1585,12,190.20,251.89,1523.31
3,ITM0000004,ORD000002,PRD0556,Ambrane Cycle Standard,Sports & Fitness,3,1107,12,398.52,631.44,3088.08
4,ITM0000005,ORD000002,PRD0026,Pebble Tablet Premium,Electronics,2,114448,18,41201.28,18707.35,251389.93
...,...,...,...,...,...,...,...,...,...,...,...
99995,ITM0099996,ORD039928,PRD0852,Zebronics Car Seat Cover Classic,Automotive,2,4197,28,2350.32,1289.35,9454.97
99996,ITM0099997,ORD039929,PRD0325,Gizmore MBA Textbook Ultra,Books,2,1301,0,0.00,202.83,2399.17
99997,ITM0099998,ORD039929,PRD0992,Noise Pen Drive Classic,Office Supplies,2,6524,18,2348.64,1946.72,13449.92
99998,ITM0099999,ORD039929,PRD0070,Noise Power Bank Classic,Electronics,1,52231,18,9401.58,3504.59,58127.99


In [ ]:
items['product_id'].is_unique

False

In [ ]:
items.duplicated(subset=['order_id', 'product_id']).sum()

np.int64(103)

In [ ]:
items.duplicated(subset=['order_id', 'product_id'], keep=False).sum()

np.int64(206)

In [ ]:
items[items.duplicated(subset=['order_id', 'product_id'], keep=False)].sort_values(['order_id', 'product_id'])

,item_id,order_id,product_id,product_name,category,quantity,unit_price,gst_rate,gst_amount,discount_amount,total_price
855,ITM0000856,ORD000348,PRD0529,Zebronics Cricket Bat Pro,Sports & Fitness,2,25820,12,6196.80,6762.71,51074.09
856,ITM0000857,ORD000348,PRD0529,Zebronics Cricket Bat Pro,Sports & Fitness,3,25820,12,9295.20,430.44,86324.76
1156,ITM0001157,ORD000464,PRD0851,ITC Car Seat Cover Standard,Automotive,1,6818,28,1909.04,693.33,8033.71
1157,ITM0001158,ORD000464,PRD0851,ITC Car Seat Cover Standard,Automotive,2,6818,28,3818.08,2405.80,15048.28
2806,ITM0002807,ORD001131,PRD0126,Titan Jeans Premium,Fashion,1,9027,5,451.35,1269.36,8208.99
...,...,...,...,...,...,...,...,...,...,...,...
98183,ITM0098184,ORD039194,PRD0290,Ambrane Bedsheet Set Classic,Home & Kitchen,1,4550,12,546.00,371.73,4724.27
98694,ITM0098695,ORD039405,PRD0411,Havells Hair Oil Standard,Beauty & Health,1,1321,18,237.78,202.18,1356.60
98695,ITM0098696,ORD039405,PRD0411,Havells Hair Oil Standard,Beauty & Health,2,1321,18,475.56,78.81,3038.75
98830,ITM0098831,ORD039457,PRD0610,Berger Board Game Classic,Toys & Baby,1,473,12,56.76,93.11,436.65


In [ ]:
items.groupby(['order_id','product_id']).size().value_counts()

,count
1,99794
2,103


Finding:
- 103 order_id + product_id combinations are repeated, involving 206 line-item records. Each duplicated combination occurs exactly twice.

In [ ]:
duplicate_items = items[items.duplicated(subset=['order_id', 'product_id'], keep=False)].sort_values(['order_id', 'product_id'])
duplicate_items[
    ['order_id', 'item_id', 'product_id',
     'quantity', 'unit_price', 'total_price']
]

,order_id,item_id,product_id,quantity,unit_price,total_price
855,ORD000348,ITM0000856,PRD0529,2,25820,51074.09
856,ORD000348,ITM0000857,PRD0529,3,25820,86324.76
1156,ORD000464,ITM0001157,PRD0851,1,6818,8033.71
1157,ORD000464,ITM0001158,PRD0851,2,6818,15048.28
2806,ORD001131,ITM0002807,PRD0126,1,9027,8208.99
...,...,...,...,...,...,...
98183,ORD039194,ITM0098184,PRD0290,1,4550,4724.27
98694,ORD039405,ITM0098695,PRD0411,1,1321,1356.60
98695,ORD039405,ITM0098696,PRD0411,2,1321,3038.75
98830,ORD039457,ITM0098831,PRD0610,1,473,436.65


Observation:
- It is observed that though there are duplicate records, these records differ in terms of *quantity* and *total_price*


Referential Integrity Verification

In [ ]:
missing_order_ids = items[~items['order_id'].isin(orders['order_id'])]
len(missing_order_ids)

0

Observation:
- There are no missing order ids. Hence, the referential integrity is preserved.

### Dataset 3: customers DA.csv

In [ ]:
import pandas as pd
# Loading csv file: customers DA.csv
customers = pd.read_csv('/content/customers DA.csv')
customers.head()

,customer_id,first_name,last_name,email,phone,city,state,pincode,gender,age,segment,registration_date,last_login_date,is_verified,is_active,total_orders,total_spent
0,CUST00001,Kunal,Gupta,kunal.gupta223@gmail.com,9117912889,Guwahati,Assam,314040,Male,54,New,30-07-2024,10-04-2025,0,1,7,1028341.51
1,CUST00002,Aryan,Gupta,aryan.gupta336@yahoo.in,9271550550,Navi Mumbai,Maharashtra,799535,Female,26,Budget,05-07-2023,03-05-2024,1,1,1,20337.39
2,CUST00003,Akash,Chatterjee,akash.chatterjee584@hotmail.com,9559676548,Surat,Gujarat,738196,Female,24,Regular,14-05-2022,31-10-2023,1,1,7,265688.26
3,CUST00004,Madhuri,Walia,madhuri.walia878@gmail.com,9673322422,Gwalior,Madhya Pradesh,238106,Female,23,Regular,21-01-2025,22-05-2025,1,0,7,173446.94
4,CUST00005,Vishal,Das,vishal.das342@yahoo.in,9920855521,Dhanbad,Jharkhand,783291,Female,59,Regular,06-06-2023,29-03-2024,1,0,4,239808.60


In [ ]:
# customers dataset dimension
customers.shape

(10000, 17)

- Number of Records = 10000
- Number of Attributes = 17

In [ ]:
# customers dataset summary
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        10000 non-null  object 
 1   first_name         10000 non-null  object 
 2   last_name          10000 non-null  object 
 3   email              10000 non-null  object 
 4   phone              10000 non-null  int64  
 5   city               10000 non-null  object 
 6   state              10000 non-null  object 
 7   pincode            10000 non-null  int64  
 8   gender             10000 non-null  object 
 9   age                10000 non-null  int64  
 10  segment            10000 non-null  object 
 11  registration_date  10000 non-null  object 
 12  last_login_date    10000 non-null  object 
 13  is_verified        10000 non-null  int64  
 14  is_active          10000 non-null  int64  
 15  total_orders       10000 non-null  int64  
 16  total_spent        1000

Observation:
- No missing / null values present in customers DA.csv
- Attribute Dtypes of *registration_date* and *last_login_date* needs to be converted from _object_ to _datetime_.

In [ ]:
# dtypes conversion from object to datetime
customers['registration_date'] = pd.to_datetime(customers['registration_date'], format = '%d-%m-%Y')
customers['last_login_date'] = pd.to_datetime(customers['last_login_date'], format = '%d-%m-%Y')

print('Data type of registration_date:', customers.registration_date.dtype)
print('Data type of last_login_date:', customers.last_login_date.dtype)

Data type of registration_date: datetime64[ns]
Data type of last_login_date: datetime64[ns]


In [ ]:
# customer_id uniqueness check
customers['customer_id'].is_unique

True

- *customer_id* is unique.

In [ ]:
# Are there customers with total_orders = 0?
len(customers[customers['total_orders'] == 0])

457

Finding:
- There are 457 customers whose total_orders = 0

In [ ]:
# value_count for customer segment
customers['segment'].value_counts(dropna=False)

,count
segment,
Regular,4026
Budget,2505
Premium,1501
New,1488
Inactive,480


In [ ]:
customers['segment'].value_counts(dropna=False, normalize=True)

,proportion
segment,
Regular,0.4026
Budget,0.2505
Premium,0.1501
New,0.1488
Inactive,0.0480


Observation:
- Customers belong to different segments such as *Regular, Budget, Premium, New,* and *Inactive*.
- Such that 40.26% are *Regular* customers, 25.05% are *Budget* customers, 15.01% are *Premium* members, and proportions of *New* and *Inactive* cutomers are 14.88% and 4.8% respectively.

In [ ]:
# value_count for is_verified
customers['is_verified'].value_counts(dropna=False)

,count
is_verified,
1,8418
0,1582


### Dataset 4: products DA.csv

In [ ]:
# Loading csv file: products DA.csv
products = pd.read_csv('/content/products DA.csv')
products.head()

,product_id,product_name,category,subcategory,brand,sku,mrp,selling_price,cost_price,gst_rate,hsn_code,weight_grams,supplier_id,rating,review_count,is_active,launch_date
0,PRD0001,Usha Smartphone Elite,Electronics,Smartphone,Usha,SKU-ELE-00001,92028,78958,55738,18,1144,250,SUP0198,4.3,3655,1,29-01-2025
1,PRD0002,Gizmore Smartphone Premium,Electronics,Smartphone,Gizmore,SKU-ELE-00002,91470,65917,48052,18,2857,2000,SUP0140,3.6,2321,1,17-05-2024
2,PRD0003,Fire-Boltt Smartphone Max,Electronics,Smartphone,Fire-Boltt,SKU-ELE-00003,148029,118800,68913,18,2349,250,SUP0113,3.5,4412,1,28-12-2024
3,PRD0004,Godrej Smartphone Elite,Electronics,Smartphone,Godrej,SKU-ELE-00004,152903,113203,80055,18,8130,250,SUP0068,4.5,1264,1,04-05-2023
4,PRD0005,HDFC Smartphone Elite,Electronics,Smartphone,HDFC,SKU-ELE-00005,18715,16226,8398,18,4729,1000,SUP0088,3.9,3700,1,27-11-2023


In [ ]:
# products dataset dimensions
products.shape

(1000, 17)

- Number of Records = 1000
- Number of Attributes = 17

In [ ]:
# products dataset summary
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   product_id     1000 non-null   object 
 1   product_name   1000 non-null   object 
 2   category       1000 non-null   object 
 3   subcategory    1000 non-null   object 
 4   brand          1000 non-null   object 
 5   sku            1000 non-null   object 
 6   mrp            1000 non-null   int64  
 7   selling_price  1000 non-null   int64  
 8   cost_price     1000 non-null   int64  
 9   gst_rate       1000 non-null   int64  
 10  hsn_code       1000 non-null   int64  
 11  weight_grams   1000 non-null   int64  
 12  supplier_id    1000 non-null   object 
 13  rating         1000 non-null   float64
 14  review_count   1000 non-null   int64  
 15  is_active      1000 non-null   int64  
 16  launch_date    1000 non-null   object 
dtypes: float64(1), int64(8), object(8)
memory usage: 132.

Observation:
- No missing / null values present in products dataset.
- *launch_date* to be converted to datetime Dtype.

In [ ]:
# convert laundh_date into datatime dtype
products['launch_date'] = pd.to_datetime(products['launch_date'], format = '%d-%m-%Y')
print('Data type of launch_date:', products.launch_date.dtype)

Data type of launch_date: datetime64[ns]


In [ ]:
# checking supplier_id uniqueness
products['supplier_id'].is_unique

False

In [ ]:
# checking product_id uniqueness
products['product_id'].is_unique

True

- *product_id* is unique.

In [ ]:
# count duplicated values in supplier_id
products['supplier_id'].duplicated(keep=False).sum()

np.int64(993)

In [ ]:
# diaplay duplicated values in supplier_id
products[products.duplicated(subset=['supplier_id'], keep=False)].sort_values('supplier_id')

,product_id,product_name,category,subcategory,brand,sku,mrp,selling_price,cost_price,gst_rate,hsn_code,weight_grams,supplier_id,rating,review_count,is_active,launch_date
26,PRD0027,Crossbeats Tablet Pro,Electronics,Tablet,Crossbeats,SKU-ELE-00027,128476,109124,70330,18,2930,5000,SUP0001,3.2,1687,1,2023-04-01
99,PRD0100,Finolex Router Lite,Electronics,Router,Finolex,SKU-ELE-00100,132167,110412,67728,18,4308,3000,SUP0001,3.4,2659,1,2025-06-20
55,PRD0056,Bajaj Electricals Bluetooth Speaker Elite,Electronics,Bluetooth Speaker,Bajaj Electricals,SKU-ELE-00056,100776,78787,46753,18,7745,5000,SUP0001,3.9,2052,1,2025-02-05
86,PRD0087,Hero Camera Plus,Electronics,Camera,Hero,SKU-ELE-00087,81589,73114,43660,18,5445,1500,SUP0001,4.7,161,1,2025-02-12
13,PRD0014,HDFC Laptop Base,Electronics,Laptop,HDFC,SKU-ELE-00014,29766,21765,14032,18,4485,100,SUP0001,4.1,4457,1,2024-07-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
431,PRD0432,Pidilite Sunscreen Max,Beauty & Health,Sunscreen,Pidilite,SKU-BEA-00432,3437,2664,1876,18,4614,1500,SUP0200,3.7,4769,1,2023-09-11
405,PRD0406,Bajaj Face Cream Classic,Beauty & Health,Face Cream,Bajaj,SKU-BEA-00406,588,445,312,18,1175,2000,SUP0200,4.4,1122,1,2024-05-13
463,PRD0464,Wipro Essential Oil Lite,Beauty & Health,Essential Oil,Wipro,SKU-BEA-00464,1531,1239,913,18,3384,750,SUP0200,3.7,261,0,2023-08-11
478,PRD0479,Portronics Kajal Plus,Beauty & Health,Kajal,Portronics,SKU-BEA-00479,803,725,466,18,1414,500,SUP0200,4.9,2938,1,2025-02-19


In [ ]:
# get unique category and subcategory
products['category'].unique()

array(['Electronics', 'Fashion', 'Home & Kitchen', 'Books',
       'Beauty & Health', 'Sports & Fitness', 'Toys & Baby', 'Grocery',
       'Automotive', 'Office Supplies'], dtype=object)

In [ ]:
# display actual values category and corresponding subcategory as dataframe
pd.DataFrame(products.groupby('category')['subcategory'].unique())

,subcategory
category,
Automotive,"[Car Cover, Bike Helmet, Car Perfume, Tyre Inf..."
Beauty & Health,"[Face Cream, Hair Oil, Shampoo, Sunscreen, Mul..."
Books,"[Fiction Novel, UPSC Guide, MBA Textbook, Cook..."
Electronics,"[Smartphone, Laptop, Tablet, Smart TV, Headpho..."
Fashion,"[Men Kurta, Women Saree, Jeans, T-Shirt, Ethni..."
Grocery,"[Basmati Rice, Organic Dal, Desi Ghee, Cold Pr..."
Home & Kitchen,"[Pressure Cooker, Mixer Grinder, Rice Cooker, ..."
Office Supplies,"[Notebook Set, Pen Pack, Printer Paper, File F..."
Sports & Fitness,"[Yoga Mat, Dumbbells, Cricket Bat, Badminton R..."


In [ ]:
# number of category present
len(products['category'].unique())

10

In [ ]:
# number of subcategory present
len(products['subcategory'].unique())

100

In [ ]:
products.groupby('category')['subcategory'].value_counts()

category     subcategory       
Automotive   Bike Helmet           10
             Bike Lock             10
             Car Cover             10
             Car Perfume           10
             Car Seat Cover        10
                                   ..
Toys & Baby  Educational Toy       10
             Puzzle Set            10
             Rattle Set            10
             Remote Control Car    10
             Soft Toy              10
Name: count, Length: 100, dtype: int64

Observation:
- There are 10 product categories.
- Each cateogry consist of 10 subcategoies each (total number of subcategories = 100)
- Each subcategory consist of 10 products each (total number of products = 1000)

In [ ]:
# value_counts for is_active
products['is_active'].value_counts(dropna=False)

,count
is_active,
1,921
0,79


In [ ]:
products['is_active'].value_counts(dropna=False, normalize=True)

,proportion
is_active,
1,0.921
0,0.079


Finding:
- 92.1% of the products are active while, 7.9% of items are inactive.
- Which means out of 1000 products: 921 products are active and 79 products are inactive

### Dataset 5: payments DA.csv

In [ ]:
import pandas as pd
# Loading csv file: payments DA.csv
payments = pd.read_csv('/content/payments DA.csv')
payments.head()

,payment_id,order_id,customer_id,payment_date,payment_time,payment_method,amount,status,transaction_id,bank_name,gateway,refund_amount,refund_date
0,PAY000001,ORD000001,CUST02946,22-05-2025,17:10:00,Cash on Delivery,100984.19,Success,TXN5110698474,SBI,Google Pay,0,NaN
1,PAY000002,ORD000002,CUST04232,16-10-2023,05:32:00,UPI,254478.01,Success,TXN4067460377,IndusInd,Google Pay,0,NaN
2,PAY000003,ORD000003,CUST08078,08-10-2023,12:32:00,Debit Card,32557.06,Success,TXN7056710694,Axis Bank,PhonePe,0,NaN
3,PAY000004,ORD000004,CUST09612,02-10-2023,08:05:00,UPI,24836.94,Success,TXN7460208257,Axis Bank,PhonePe,0,NaN
4,PAY000005,ORD000005,CUST03065,15-06-2024,20:19:00,UPI,156689.29,Success,TXN5835043807,PNB,Razorpay,0,NaN


In [ ]:
# payments dataset dimensions
payments.shape

(50000, 13)

- Number of Records = 9462
- Number of Attributes = 13

In [ ]:
# payments dataset summary
payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   payment_id      50000 non-null  object 
 1   order_id        50000 non-null  object 
 2   customer_id     50000 non-null  object 
 3   payment_date    50000 non-null  object 
 4   payment_time    50000 non-null  object 
 5   payment_method  50000 non-null  object 
 6   amount          50000 non-null  float64
 7   status          50000 non-null  object 
 8   transaction_id  50000 non-null  object 
 9   bank_name       50000 non-null  object 
 10  gateway         50000 non-null  object 
 11  refund_amount   50000 non-null  int64  
 12  refund_date     0 non-null      float64
dtypes: float64(2), int64(1), object(10)
memory usage: 5.0+ MB


Observation:
- *refund_date* is consisting of only null values.
- *payment_date* to be converted to datatime Dtype.
- *payment_time* to be parsed into time object.
- Columns containing 1 NaN value: *payment_method, amount, status, transaction_id, bank_name, gateway, refund_amount*
- null value in *status* is not reflected here. WHY?

In [ ]:
# convert payment_date to datatime dtype
payments['payment_date'] = pd.to_datetime(payments['payment_date'], format = '%d-%m-%Y')
print('Data type of payment_date:', payments['payment_date'].dtype)

# convert payment_time to time object
payments['payment_time'] = pd.to_datetime(payments['payment_time'], format = 'mixed').dt.time
print('Data type of payment_time:', payments['payment_time'].dtype)

Data type of payment_date: datetime64[ns]
Data type of payment_time: object


In [ ]:
# checking whether the payment_time has been sucessfully parsed into time object
type(payments['payment_time'].iloc[0])

datetime.time

In [ ]:
# checking the unique values in refund_amount
payments['refund_amount'].unique()

array([0])

In [ ]:
payments['refund_amount'].value_counts(dropna=False)

,count
refund_amount,
0,50000


Observation:
- All non-null *refund_amount* are 0.0; one record has a missing refund amount.
- Thus, justifying the null values in the *refund_date*.

In [ ]:
payments['bank_name'].value_counts(dropna=False)

,count
bank_name,
HDFC Bank,5054
IndusInd,5044
Bank of Baroda,5039
Canara Bank,5026
SBI,5019
Kotak Mahindra,5004
ICICI Bank,4972
Axis Bank,4970
Yes Bank,4955


In [ ]:
payments['gateway'].value_counts(dropna=False)

,count
gateway,
BillDesk,7222
Google Pay,7182
PhonePe,7170
Razorpay,7126
PayU,7121
CCAvenue,7118
Paytm,7061


In [ ]:
payments['status'].unique()

array(['Success', 'Failed'], dtype=object)

In [ ]:
# count failed vs sucess transactions - status
payments['status'].value_counts(dropna=False)

,count
status,
Success,48252
Failed,1748


In [ ]:
# caluculate failure rate using status
payments['status'].value_counts(dropna=False, normalize=True).mul(100).round(2)

,proportion
status,
Success,96.5
Failed,3.5


Finding:
- Payment status percentage: Success = 96.55%, Faliure rate = 3.43%, NaN = 0.01

In [ ]:
# checking uniqueness of payment_id
payments['payment_id'].is_unique

True

- *payment_id* is unique.

In [ ]:
payments[payments['status'].isna()]

,payment_id,order_id,customer_id,payment_date,payment_time,payment_method,amount,status,transaction_id,bank_name,gateway,refund_amount,refund_date


Finding:
- One record is identified (payment_id = PAY028383) to have NaN values for multiple attributes (payment_method,	amount,	status,	transaction_id,	bank_name,	gateway,	refund_amount,	refund_date)

In [ ]:
# paymemt->order referential integrity
missing_payment_orders = payments[~payments['order_id'].isin(orders['order_id'])]
len(missing_payment_orders)

0

Observation:

All payment records reference valid orders.

### Dataset 6: returns DA.csv

In [ ]:
import pandas as pd
# Loading csv file: reutrns DA.csv
returns = pd.read_csv('/content/returns DA.csv')
returns.head()

,return_id,order_id,customer_id,return_date,reason,return_amount,refund_status,refund_date,return_condition,remarks
0,RET00001,ORD000015,CUST05470,23-12-2024,Size Issue,24167.33,Pending,30-12-2024,Damaged,Customer initiated return within 7 day window
1,RET00002,ORD000017,CUST07581,01-11-2023,Defective Product,11698.27,Refunded,06-11-2023,Damaged,Customer initiated return within 7 day window
2,RET00003,ORD000038,CUST05691,01-05-2025,Wrong Product Delivered,11147.74,Refunded,06-05-2025,Damaged,Customer initiated return within 7 day window
3,RET00004,ORD000054,CUST03170,20-06-2025,Duplicate Order,9284.55,Refunded,24-06-2025,Acceptable,Customer initiated return within 7 day window
4,RET00005,ORD000072,CUST09318,16-03-2025,Duplicate Order,36912.59,Refunded,23-03-2025,Good,Customer initiated return within 7 day window


In [ ]:
# returns dataset dimension
returns.shape

(10000, 10)

- Number of Records = 10000
- Number of Attributes = 10

In [ ]:
# returns dataset summary
returns.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   return_id         10000 non-null  object 
 1   order_id          10000 non-null  object 
 2   customer_id       10000 non-null  object 
 3   return_date       10000 non-null  object 
 4   reason            10000 non-null  object 
 5   return_amount     10000 non-null  float64
 6   refund_status     10000 non-null  object 
 7   refund_date       10000 non-null  object 
 8   return_condition  10000 non-null  object 
 9   remarks           10000 non-null  object 
dtypes: float64(1), object(9)
memory usage: 781.4+ KB


Observation:
- No null values present in the returns dataset.
- Attritubes to be converted from object to datetime dtype: *return_date, refund_date*

In [ ]:
# checking uniqueness of return_id
returns['return_id'].is_unique

True

- *return_id*  is unique.

In [ ]:
# convert return_date and refund_date to datatime dtype
returns['return_date'] = pd.to_datetime(
    returns['return_date'],
    format='%d-%m-%Y'
)
print('return_date dtype:',returns['return_date'].dtype)

returns['refund_date'] = pd.to_datetime(
    returns['refund_date'],
    format='%d-%m-%Y'
)
print('refund_date dtype:',returns['refund_date'].dtype)

return_date dtype: datetime64[ns]
refund_date dtype: datetime64[ns]


In [ ]:
# referential integrity
missing_return_orders = returns[
    returns['order_id'].notna() &
    ~returns['order_id'].isin(orders['order_id'])
]

len(missing_return_orders)

0

Finding:
- 4238 return records does not have a corresponding order_id.

### Dataset 7: inventory DA.csv

In [ ]:
# Load csv file: inventory DA.csv
inventory = pd.read_csv('/content/inventory DA.csv')
inventory.head()

,inventory_id,product_id,warehouse_location,quantity_available,quantity_reserved,reorder_level,reorder_quantity,last_restocked_date,unit_cost,total_inventory_value,status
0,INV0001,PRD0001,Hyderabad-WH4,13,4,49,245,11-09-2023,55738,724594,Low Stock
1,INV0002,PRD0002,Kolkata-WH6,368,14,50,250,10-03-2025,48052,17683136,In Stock
2,INV0003,PRD0003,Delhi-WH2,196,19,44,220,22-11-2024,68913,13506948,In Stock
3,INV0004,PRD0004,Delhi-WH2,432,10,19,57,10-11-2024,80055,34583760,In Stock
4,INV0005,PRD0005,Bengaluru-WH3,404,16,44,88,05-08-2024,8398,3392792,In Stock


In [ ]:
# inventory dataset dimension
inventory.shape

(1000, 11)

- Number of Records = 1000
- Number of Attributes = 11

In [ ]:
# inventory dataset summary
inventory.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   inventory_id           1000 non-null   object
 1   product_id             1000 non-null   object
 2   warehouse_location     1000 non-null   object
 3   quantity_available     1000 non-null   int64 
 4   quantity_reserved      1000 non-null   int64 
 5   reorder_level          1000 non-null   int64 
 6   reorder_quantity       1000 non-null   int64 
 7   last_restocked_date    1000 non-null   object
 8   unit_cost              1000 non-null   int64 
 9   total_inventory_value  1000 non-null   int64 
 10  status                 1000 non-null   object
dtypes: int64(6), object(5)
memory usage: 86.1+ KB


In [ ]:
inventory.isna().sum()

,0
inventory_id,0
product_id,0
warehouse_location,0
quantity_available,0
quantity_reserved,0
reorder_level,0
reorder_quantity,0
last_restocked_date,0
unit_cost,0
total_inventory_value,0


Observation:
- No null values present in *inventory* dataset
- *last_restocked_date* to be converted to datatime dtype.

In [ ]:
# converting last_restocked_date to datetime dtype
inventory['last_restocked_date'] = pd.to_datetime(
    inventory['last_restocked_date'],
    format='%d-%m-%Y'
)
print('Data type of last_restocked_date:', inventory['last_restocked_date'].dtype)

Data type of last_restocked_date: datetime64[ns]


In [ ]:
# check for uniqueness
print('Uniqueness of product_id:',inventory['product_id'].is_unique)
print('Uniqueness of inventory_id:', inventory['inventory_id'].is_unique)

Uniqueness of product_id: True
Uniqueness of inventory_id: True


Finding:
- Both *product_id* and *inventory_id* are unique.

In [ ]:
# How many products are in Out of Stock status?
inventory['status'].value_counts(dropna=False)

,count
status,
In Stock,922
Low Stock,76
Out of Stock,2


Finding:
- 2 items are out of stock.

In [ ]:
inventory[
    ~inventory['product_id'].isin(products['product_id'])
]

,inventory_id,product_id,warehouse_location,quantity_available,quantity_reserved,reorder_level,reorder_quantity,last_restocked_date,unit_cost,total_inventory_value,status


Findings:
- Both inventory dataset and products dataset have same product_ids but they are not reflected here. WHY?

### Dataset 8: suppliers DA.csv

In [ ]:
# Load csv file: suppliers DA.csv
suppliers = pd.read_csv('/content/suppliers DA.csv')
suppliers.head()

,supplier_id,supplier_name,contact_person,email,phone,city,state,pincode,category,gstin,payment_terms_days,rating,created_date,is_active
0,SUP0001,Iyer Elec Pvt Ltd,Harish Iyer,contact@iyerele.com,9362950628,Delhi,Delhi,334053,Electronics,14TRXCK1488C4Z9,15,4.1,22-05-2023,1
1,SUP0002,Sharma Offi Pvt Ltd,Mohit Sharma,contact@sharmaoff.com,9914763202,Ahmedabad,Gujarat,267414,Office Supplies,32KHFTC7224C6Z5,15,4.5,25-03-2023,1
2,SUP0003,Choudhury Beau Pvt Ltd,Nandini Choudhury,contact@choudhurybea.com,9764130526,Pune,Maharashtra,479201,Beauty & Health,28EBRUZ4814W7Z5,60,4.3,27-12-2022,1
3,SUP0004,Choudhury Fash Pvt Ltd,Vikram Choudhury,contact@choudhuryfas.com,9283758720,Chennai,Tamil Nadu,660086,Fashion,33GMHYR4598R1Z4,15,4.6,15-01-2023,1
4,SUP0005,Patil Spor Pvt Ltd,Kiran Patil,contact@patilspo.com,9636045484,Jaipur,Rajasthan,514850,Sports & Fitness,30LGGOG8019X7Z6,30,5.0,02-09-2022,1


In [ ]:
# suppliers dataset dimension
suppliers.shape

(200, 14)

- Number of Records = 200
- Number of Attributes = 14

In [ ]:
# suppliers dataset summary
suppliers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   supplier_id         200 non-null    object 
 1   supplier_name       200 non-null    object 
 2   contact_person      200 non-null    object 
 3   email               200 non-null    object 
 4   phone               200 non-null    int64  
 5   city                200 non-null    object 
 6   state               200 non-null    object 
 7   pincode             200 non-null    int64  
 8   category            200 non-null    object 
 9   gstin               200 non-null    object 
 10  payment_terms_days  200 non-null    int64  
 11  rating              200 non-null    float64
 12  created_date        200 non-null    object 
 13  is_active           200 non-null    int64  
dtypes: float64(1), int64(4), object(9)
memory usage: 22.0+ KB


In [ ]:
suppliers.isna().sum()

,0
supplier_id,0
supplier_name,0
contact_person,0
email,0
phone,0
city,0
state,0
pincode,0
category,0
gstin,0


Observation:
- No null values present in *suppliers* dataset.
- *created_date* to be changed to datetime dtype.

In [ ]:
# convert created_date to datetime dtype
suppliers['created_date'] = pd.to_datetime(
    suppliers['created_date'],
    format='%d-%m-%Y'
)
print('Data type of created_date:', suppliers['created_date'].dtype)

Data type of created_date: datetime64[ns]


In [ ]:
# check uniqueness of supplier_id
suppliers['supplier_id'].is_unique

True

In [ ]:
suppliers.describe()

,phone,pincode,payment_terms_days,rating,created_date,is_active
count,2.000000e+02,200.00000,200.000000,200.000000,200,200.000000
mean,9.545867e+09,456054.90000,36.975000,4.027000,2022-12-28 10:48:00,0.900000
min,9.101930e+09,104895.00000,15.000000,3.000000,2022-06-24 00:00:00,0.000000
25%,9.334131e+09,271254.00000,15.000000,3.500000,2022-09-19 00:00:00,1.000000
50%,9.553643e+09,467217.00000,30.000000,4.000000,2022-12-27 12:00:00,1.000000
75%,9.733708e+09,624113.00000,60.000000,4.600000,2023-04-01 06:00:00,1.000000
max,9.994254e+09,792764.00000,60.000000,5.000000,2023-06-24 00:00:00,1.000000
std,2.458111e+08,202482.34608,17.657386,0.586422,NaN,0.300753


Observation:
- With respect to rating:
> minimum rating is 3 <br> maximum rating is 5 <br> average rating is 4
- With respect to payment_terms_days:
> minimum days is 15 <br> maximum days 60 <br> average days 36

# Data Quality Report / Summary

This section summarizes the data quality assessment performed across all eight IndiaKart datasets.
The assessment covers data completeness, uniqueness, data-type validation, duplicate records,
outlier identification, referential integrity, and the specific quality checks defined in the
Project Requirements Document (PRD).

The findings below represent the final results after completing the required data-quality checks.

Dataset Overview

In [ ]:
# DATASET OVERVIEW

dataset_overview = pd.DataFrame({
    'Dataset': [
        'Orders',
        'Order Items',
        'Customers',
        'Products',
        'Payments',
        'Returns',
        'Inventory',
        'Suppliers'
    ],

    'Rows': [
        len(orders),
        len(items),
        len(customers),
        len(products),
        len(payments),
        len(returns),
        len(inventory),
        len(suppliers)
    ],

    'Columns': [
        orders.shape[1],
        items.shape[1],
        customers.shape[1],
        products.shape[1],
        payments.shape[1],
        returns.shape[1],
        inventory.shape[1],
        suppliers.shape[1]
    ]
})

dataset_overview

,Dataset,Rows,Columns
0,Orders,50000,19
1,Order Items,100000,11
2,Customers,10000,17
3,Products,1000,17
4,Payments,50000,13
5,Returns,10000,10
6,Inventory,1000,11
7,Suppliers,200,14


Missing Value Summary

In [ ]:
# MISSING VALUE SUMMARY

datasets = {
    'Orders': orders,
    'Order Items': items,
    'Customers': customers,
    'Products': products,
    'Payments': payments,
    'Returns': returns,
    'Inventory': inventory,
    'Suppliers': suppliers
}

missing_summary = []

for dataset_name, df in datasets.items():
    missing_counts = df.isna().sum()

    for column, count in missing_counts.items():
        if count > 0:
            missing_summary.append({
                'Dataset': dataset_name,
                'Column': column,
                'Missing Count': count,
                'Missing %': round((count / len(df)) * 100, 2)
            })

missing_summary = pd.DataFrame(missing_summary)

missing_summary

,Dataset,Column,Missing Count,Missing %
0,Orders,delivered_date,17501,35.0
1,Payments,refund_date,50000,100.0


Final Data Quality Findings

In [ ]:
# FINAL DATA QUALITY FINDINGS

# Orders

orders_duplicate_ids = orders['order_id'].duplicated().sum()

orders_invalid_dates = (
    orders['delivered_date'].notna() &
    (orders['delivered_date'] <= orders['order_date'])
).sum()

orders_missing_delivery = orders['delivered_date'].isna().sum()

orders_high_value = (orders['final_amount'] > 500000).sum()

tracking_id_duplicate_groups = (
    orders.groupby('tracking_id')['order_id']
    .nunique()
    .gt(1)
    .sum()
)


# Order_Items

duplicate_order_product = (
    items.duplicated(subset=['order_id', 'product_id']).sum()
)

duplicate_order_product_records = (
    items.duplicated(subset=['order_id', 'product_id'], keep=False).sum()
)

missing_item_order_ids = (
    ~items['order_id'].isin(orders['order_id'])
).sum()


# Customers

zero_order_customers = (
    customers['total_orders'] == 0
).sum()


# Payments

payment_failure_count = payments['status'].astype(str).str.strip().str.lower().isin(
    ['failed', 'failure']
).sum()

payment_success_count = payments['status'].astype(str).str.strip().str.lower().eq(
    'success'
).sum()

payment_failure_rate = (
    payment_failure_count / len(payments) * 100
)

payment_missing_orders = (
    ~payments['order_id'].isin(orders['order_id'])
).sum()


# Returns

missing_return_orders = returns[
    returns['order_id'].notna() &
    ~returns['order_id'].isin(orders['order_id'])
].shape[0]


# Inventory

out_of_stock_count = (
    inventory['status'].astype(str).str.strip().str.lower()
    .eq('out of stock')
).sum()


# FINAL FINDINGS TABLE

quality_findings = pd.DataFrame([

    # Orders
    {
        'Dataset': 'Orders',
        'Check': 'Duplicate order_id',
        'Result': orders_duplicate_ids,
        'Unit': 'duplicates',
        'Status': 'Pass' if orders_duplicate_ids == 0 else 'Issue'
    },

    {
        'Dataset': 'Orders',
        'Check': 'Missing delivered_date',
        'Result': orders_missing_delivery,
        'Unit': 'records',
        'Status': 'Reviewed'
    },

    {
        'Dataset': 'Orders',
        'Check': 'Invalid delivered_date',
        'Result': orders_invalid_dates,
        'Unit': 'records',
        'Status': 'Pass' if orders_invalid_dates == 0 else 'Issue'
    },

    {
        'Dataset': 'Orders',
        'Check': 'final_amount > ₹5 lakh',
        'Result': orders_high_value,
        'Unit': 'orders',
        'Status': 'Reviewed' if orders_high_value > 0 else 'Pass'
    },

    {
        'Dataset': 'Orders',
        'Check': 'Tracking ID linked to multiple orders',
        'Result': tracking_id_duplicate_groups,
        'Unit': 'tracking IDs',
        'Status': 'Reviewed' if tracking_id_duplicate_groups > 0 else 'Pass'
    },

    # Order Items
    {
        'Dataset': 'Order Items',
        'Check': 'Duplicate order_id + product_id combinations',
        'Result': duplicate_order_product,
        'Unit': 'combinations',
        'Status': 'Reviewed' if duplicate_order_product > 0 else 'Pass'
    },

    {
        'Dataset': 'Order Items',
        'Check': 'Records involved in repeated combinations',
        'Result': duplicate_order_product_records,
        'Unit': 'records',
        'Status': 'Reviewed' if duplicate_order_product_records > 0 else 'Pass'
    },

    {
        'Dataset': 'Order Items',
        'Check': 'Order IDs missing from Orders',
        'Result': missing_item_order_ids,
        'Unit': 'records',
        'Status': 'Pass' if missing_item_order_ids == 0 else 'Issue'
    },

    # Customers
    {
        'Dataset': 'Customers',
        'Check': 'Customers with total_orders = 0',
        'Result': zero_order_customers,
        'Unit': 'customers',
        'Status': 'Not an issue'
    },

    # Payments
    {
        'Dataset': 'Payments',
        'Check': 'Successful payments',
        'Result': payment_success_count,
        'Unit': 'payments',
        'Status': 'Informational'
    },

    {
        'Dataset': 'Payments',
        'Check': 'Failed payments',
        'Result': payment_failure_count,
        'Unit': 'payments',
        'Status': 'Reviewed'
    },

    {
        'Dataset': 'Payments',
        'Check': 'Payment failure rate',
        'Result': round(payment_failure_rate, 2),
        'Unit': '%',
        'Status': 'Reviewed'
    },

    {
        'Dataset': 'Payments',
        'Check': 'Payment records with invalid order IDs',
        'Result': payment_missing_orders,
        'Unit': 'records',
        'Status': 'Pass' if payment_missing_orders == 0 else 'Issue'
    },

    # Returns
    {
        'Dataset': 'Returns',
        'Check': 'Returns without matching order ID',
        'Result': missing_return_orders,
        'Unit': 'records',
        'Status': 'Reviewed' if missing_return_orders > 0 else 'Pass'
    },

    # Inventory
    {
        'Dataset': 'Inventory',
        'Check': 'Out of Stock products',
        'Result': out_of_stock_count,
        'Unit': 'products',
        'Status': 'Reviewed'
    }
])

quality_findings

,Dataset,Check,Result,Unit,Status
0,Orders,Duplicate order_id,0.0,duplicates,Pass
1,Orders,Missing delivered_date,17501.0,records,Reviewed
2,Orders,Invalid delivered_date,0.0,records,Pass
3,Orders,final_amount > ₹5 lakh,312.0,orders,Reviewed
4,Orders,Tracking ID linked to multiple orders,1.0,tracking IDs,Reviewed
5,Order Items,Duplicate order_id + product_id combinations,103.0,combinations,Reviewed
6,Order Items,Records involved in repeated combinations,206.0,records,Reviewed
7,Order Items,Order IDs missing from Orders,0.0,records,Pass
8,Customers,Customers with total_orders = 0,457.0,customers,Not an issue
9,Payments,Successful payments,48252.0,payments,Informational


## Create Cleaned Datasets

In [ ]:
# creating cleaned datasets

orders.to_csv('cleaned_orders.csv', index=False)
items.to_csv('cleaned_order_items.csv', index=False)
customers.to_csv('cleaned_customers.csv', index=False)
products.to_csv('cleaned_products.csv', index=False)
payments.to_csv('cleaned_payments.csv', index=False)
returns.to_csv('cleaned_returns.csv', index=False)
inventory.to_csv('cleaned_inventory.csv', index=False)
suppliers.to_csv('cleaned_suppliers.csv', index=False)

print('csv files created sucessfully. Please download them to local storage')

csv files created sucessfully. Please download them to local storage


Download the cleaned datasets

In [ ]:
# download the cleaned datasets

from google.colab import files

files.download('cleaned_orders.csv')
files.download('cleaned_order_items.csv')
files.download('cleaned_customers.csv')
files.download('cleaned_products.csv')
files.download('cleaned_payments.csv')
files.download('cleaned_returns.csv')
files.download('cleaned_inventory.csv')
files.download('cleaned_suppliers.csv')

print('Download complete!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download complete!
